# Barren Plateau Analysis

Investigate gradient behavior and barren plateau phenomena across different approaches.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Load Training Histories

In [ ]:
# Load training history JSON files
results_dir = Path('../results')

histories = {'baseline': [], 'layerwise': [], 'local_cost': []}

for approach in histories.keys():
    json_files = list(results_dir.glob(f'{approach}_*.json'))
    for file in json_files[:5]:  # Load first 5 runs
        with open(file) as f:
            histories[approach].append(json.load(f))

print(f"Loaded histories:")
for approach, hist_list in histories.items():
    print(f"  {approach}: {len(hist_list)} runs")

## 2. Gradient Norm Trajectories

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (approach, hist_list) in enumerate(histories.items()):
    if hist_list:
        ax = axes[idx]
        
        for hist in hist_list:
            if 'gradient_norms' in hist:
                grad_norms = hist['gradient_norms']
                ax.semilogy(grad_norms, alpha=0.5, linewidth=1.5)
        
        ax.axhline(y=1e-6, color='red', linestyle='--', linewidth=2, label='BP threshold')
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('Gradient Norm (log scale)', fontsize=11)
        ax.set_title(f'{approach.replace("_", " ").title()}', fontsize=13)
        ax.legend()
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Gradient Variance Over Time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (approach, hist_list) in enumerate(histories.items()):
    if hist_list:
        ax = axes[idx]
        
        for hist in hist_list:
            if 'gradient_variances' in hist:
                grad_vars = hist['gradient_variances']
                ax.semilogy(grad_vars, alpha=0.5, linewidth=1.5)
        
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('Gradient Variance (log scale)', fontsize=11)
        ax.set_title(f'{approach.replace("_", " ").title()}', fontsize=13)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Barren Plateau Detection

In [ ]:
threshold = 1e-6
bp_stats = {}

for approach, hist_list in histories.items():
    bp_count = 0
    total_runs = len(hist_list)
    
    for hist in hist_list:
        if 'has_barren_plateau' in hist and hist['has_barren_plateau']:
            bp_count += 1
    
    bp_stats[approach] = (bp_count / total_runs * 100) if total_runs > 0 else 0

# Plot
plt.figure(figsize=(10, 6))
bars = plt.bar(bp_stats.keys(), bp_stats.values(), 
               color=['salmon', 'lightcoral', 'indianred'], edgecolor='black')
plt.xlabel('Approach', fontsize=12)
plt.ylabel('Barren Plateau Occurrence (%)', fontsize=12)
plt.title(f'Barren Plateau Detection Rate (Threshold: {threshold})', fontsize=14)
plt.ylim(0, 105)
plt.grid(alpha=0.3, axis='y')

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

print("\nBarren Plateau Detection:")
for approach, rate in bp_stats.items():
    print(f"  {approach}: {rate:.1f}% of runs")

## 5. Gradient Distribution Analysis

In [ ]:
# Analyze gradient distribution at different training stages
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
stages = ['Early (0-10)', 'Mid (20-30)', 'Late (40-50)']
epochs_ranges = [(0, 10), (20, 30), (40, 50)]

for idx, (start, end) in enumerate(epochs_ranges):
    ax = axes[idx]
    
    for approach, hist_list in histories.items():
        all_grads = []
        for hist in hist_list:
            if 'gradient_norms' in hist:
                grads = hist['gradient_norms'][start:end]
                all_grads.extend(grads)
        
        if all_grads:
            ax.hist(np.log10(all_grads), bins=30, alpha=0.5, label=approach)
    
    ax.axvline(x=np.log10(1e-6), color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('log₁₀(Gradient Norm)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{stages[idx]} Epochs', fontsize=12)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Loss vs Gradient Correlation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (approach, hist_list) in enumerate(histories.items()):
    if hist_list:
        ax = axes[idx]
        
        for hist in hist_list:
            if 'train_loss' in hist and 'gradient_norms' in hist:
                loss = hist['train_loss']
                grads = hist['gradient_norms'][:len(loss)]
                ax.scatter(loss, grads, alpha=0.3, s=20)
        
        ax.set_xlabel('Training Loss', fontsize=11)
        ax.set_ylabel('Gradient Norm', fontsize=11)
        ax.set_yscale('log')
        ax.set_title(f'{approach.replace("_", " ").title()}', fontsize=13)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Conclusions

Key findings:
- Which approach best avoids barren plateaus?
- How do gradient norms evolve during training?
- Impact of local cost functions on gradient flow
- Effectiveness of layerwise training strategy